# Setup

**MESSENGER conjunction ICME analysis**

Find cataloged ICME starts at MESSENGER and the partners in the last column of
the supplied conjunction table. Earth uses both Wind/HELIO4CAST and the SINP
merged list. Run top-to-bottom from the Shock-and-Awe repository with the
`icme3.12-metal` kernel. Inputs and results are local to this repository.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

In [2]:
REPO = Path.cwd()
INPUT_DIR = REPO / "Conjunctions" / "inputs"
CATALOG_DIR = REPO / "HELIO4CAST"
RESULTS_DIR = REPO / "Conjunctions" / "results"
TOLERANCE = pd.Timedelta(hours=24)
AU_KM = 149_597_870.7

assert INPUT_DIR.is_dir(), "Start this notebook from the Shock-and-Awe repository."
assert TOLERANCE > pd.Timedelta(0)

# Inputs and time definitions

The conjunction CSV and the `messenger_conjunctions` workbook sheet are two
representations of the same daily table; verify them before matching. Its last
column lists partners inside the supplied 20° cone, separated by semicolons.
Use those listed partners directly. Dates are midnight daily samples, treated as
UTC (an explicit assumption for the timezone-naive input), not event times or
continuous observing intervals. No ephemerides are recomputed or interpolated.

**Terminology:** our ICME `start_time` corresponds to HELIO4CAST
`mo_start_time`, the magnetic-obstacle start. HELIO4CAST `icme_start_time`
is the earlier shock/disturbance arrival and is retained only as context.
SINP uses `T_start` for the ICME start and `T_shock` for the shock. Missing
`T_start` stays missing; do not substitute `T_shock` or the solar-source
`T_source`. SINP times are parsed as UTC, consistent with its source lists.

HELIO4CAST v2.3: [catalog](https://helioforecast.space/icmecat),
[authors' CSV](https://raw.githubusercontent.com/cmoestl/heliocats/master/icmecat/HELIO4CAST_ICMECAT_v23.csv).
The saved parameter header describes units and frames. Positions are HEEQ;
magnetic components use RTN for Wind/STEREO and SCEQ for MESSENGER/VEX.
Only times and supplied geometry are used here; no magnetic vectors or units
are converted. Missing plasma parameters at MESSENGER/VEX are left untouched.

SINP snapshot: `ICME_list/out/site/merged_icme_short.csv`. Its producer is
`ICME_list/Control Center.ipynb` and `processing/catalog_merging.py`:
`cat` digits 1/2/3 denote Richardson–Cane / CCMC / IKI contributions.
CCMC-only (`cat=2`) rows have shock arrivals but no ICME-body start/end.
The workbook's earlier `icmes_Earth+Merc` selection is a comparison source,
not a pre-established MESSENGER association.

In [3]:
input_manifest = json.loads((INPUT_DIR / "input_provenance.json").read_text())
catalog_manifest = json.loads((CATALOG_DIR / "download_metadata.json").read_text())
for name, metadata in input_manifest.items():
    assert hashlib.sha256((INPUT_DIR / name).read_bytes()).hexdigest() == metadata["sha256"]
for name, digest in catalog_manifest["sha256"].items():
    assert hashlib.sha256((CATALOG_DIR / name).read_bytes()).hexdigest() == digest

conjunctions = pd.read_csv(INPUT_DIR / "messenger_conjunctions.csv")
workbook_conjunctions = pd.read_excel(
    INPUT_DIR / "mercury_conj.xlsx", sheet_name="messenger_conjunctions"
)
for table in (conjunctions, workbook_conjunctions):
    table["Date"] = pd.to_datetime(table["Date"], utc=True)
pd.testing.assert_frame_equal(conjunctions, workbook_conjunctions, check_dtype=False)
assert conjunctions["Date"].is_unique
assert conjunctions["Date"].eq(conjunctions["Date"].dt.normalize()).all()
assert conjunctions.notna().all().all()
conjunctions.insert(0, "conjunction_id", [f"CONJ-{i + 1:04d}" for i in range(len(conjunctions))])

prior_earth = pd.read_excel(INPUT_DIR / "mercury_conj.xlsx", sheet_name="icmes_Earth+Merc")
for column in ("T_start", "T_shock", "T_end", "T_gen"):
    prior_earth[column] = pd.to_datetime(prior_earth[column], utc=True)

display(pd.DataFrame([{
    "conjunction_rows": len(conjunctions),
    "first_UTC": conjunctions["Date"].min(),
    "last_UTC": conjunctions["Date"].max(),
    "CSV_equals_XLS": True,
    "earlier_Earth_selection_rows": len(prior_earth),
}]))

,conjunction_rows,first_UTC,last_UTC,CSV_equals_XLS,earlier_Earth_selection_rows
0,605,2011-03-18 00:00:00+00:00,2015-04-13 00:00:00+00:00,True,33


## Normalize event starts without substituting shocks

Use all five downloaded 2011–2015 subsets. The downloader's year selection is
based on arrival time, while the analysis uses MO starts. Check the full saved
source as well to ensure that this distinction has not dropped a boundary event.

In [4]:
helio_tables = []
for spacecraft, metadata in catalog_manifest["spacecraft"].items():
    table = pd.read_csv(CATALOG_DIR / metadata["file"])
    assert table["sc_insitu"].eq(spacecraft).all()
    assert len(table) == metadata["events"]
    helio_tables.append(table)
helio = pd.concat(helio_tables, ignore_index=True)
for column in ("icme_start_time", "mo_start_time", "mo_end_time"):
    helio[column] = pd.to_datetime(helio[column], utc=True)
assert helio["icmecat_id"].is_unique
assert helio[["mo_start_time", "mo_end_time"]].notna().all().all()
assert (helio["mo_end_time"] >= helio["mo_start_time"]).all()

source = pd.read_csv(CATALOG_DIR / "source" / "HELIO4CAST_ICMECAT_v23.csv")
source["mo_start_time"] = pd.to_datetime(source["mo_start_time"], utc=True)
analysis_start = conjunctions["Date"].min() - TOLERANCE
analysis_end = conjunctions["Date"].max() + TOLERANCE
source_relevant = source[
    source["sc_insitu"].isin(catalog_manifest["spacecraft"])
    & source["mo_start_time"].between(analysis_start, analysis_end)
]
assert set(source_relevant["icmecat_id"]) <= set(helio["icmecat_id"])

sinp = pd.read_csv(INPUT_DIR / "merged_icme_short.csv").rename(columns={"Unnamed: 0": "source_index"})
assert sinp["source_index"].is_unique
for column in ("T_source", "T_shock", "T_start", "T_end"):
    sinp[column] = pd.to_datetime(sinp[column], utc=True)
sinp["event_id"] = "SINP-" + sinp["source_index"].astype(str)

helio_events = helio.rename(columns={
    "icmecat_id": "event_id", "sc_insitu": "observer",
    "mo_start_time": "start_time", "mo_end_time": "end_time",
    "icme_start_time": "arrival_time",
})[["event_id", "observer", "start_time", "end_time", "arrival_time"]].copy()
helio_events["catalog"] = "HELIO4CAST"
helio_events["start_field"] = "mo_start_time"
sinp_events = sinp.rename(columns={
    "T_start": "start_time", "T_end": "end_time", "T_shock": "arrival_time",
})[["event_id", "start_time", "end_time", "arrival_time", "cat"]].copy()
sinp_events["observer"] = "Earth (SINP merged)"
sinp_events["catalog"] = "SINP"
sinp_events["start_field"] = "T_start"
events = pd.concat([helio_events, sinp_events], ignore_index=True)
events["target"] = events["observer"].map({
    "MESSENGER": "Mercury", "STEREO-A": "STEREO_A", "STEREO-B": "STEREO_B",
    "VEX": "Venus", "Wind": "Earth", "Earth (SINP merged)": "Earth",
})
assert events["target"].notna().all()
sinp_missing_start = sinp[
    sinp["T_start"].isna() & sinp["T_shock"].between(analysis_start, analysis_end)
].copy()
events_in_span = events[events["start_time"].between(analysis_start, analysis_end)].copy()
display(events_in_span.groupby(["target", "catalog", "start_field"]).agg(
    events=("event_id", "size"), first_start=("start_time", "min"), last_start=("start_time", "max")
).reset_index())

,target,catalog,start_field,events,first_start,last_start
0,Earth,HELIO4CAST,mo_start_time,88,2011-03-29 23:59:00+00:00,2015-04-09 23:02:00+00:00
1,Earth,SINP,T_start,199,2011-03-29 23:00:00+00:00,2015-04-10 13:00:00+00:00
2,Mercury,HELIO4CAST,mo_start_time,64,2011-05-19 15:00:00+00:00,2015-04-08 02:04:00+00:00
3,STEREO_A,HELIO4CAST,mo_start_time,131,2011-03-19 23:34:00+00:00,2014-08-12 07:25:00+00:00
4,STEREO_B,HELIO4CAST,mo_start_time,103,2011-03-21 10:00:00+00:00,2014-09-25 17:22:00+00:00
5,Venus,HELIO4CAST,mo_start_time,58,2011-03-19 03:19:00+00:00,2014-09-26 18:04:00+00:00


## Missing SINP start times

These Earth shock entries lie inside the conjunction table's overall date span
but cannot participate in an ICME-start match. Absence of a candidate is not a
statement that no ICME occurred or that a spacecraft had continuous coverage.

In [5]:
display(sinp_missing_start[["event_id", "T_shock", "T_start", "T_end", "cat"]])

,event_id,T_shock,T_start,T_end,cat
194,SINP-967,2013-05-19 22:20:00+00:00,NaT,NaT,2
199,SINP-972,2013-06-23 03:45:00+00:00,NaT,NaT,2
201,SINP-974,2013-06-30 10:40:00+00:00,NaT,NaT,2
205,SINP-978,2013-07-13 01:00:00+00:00,NaT,NaT,2
211,SINP-984,2013-09-02 01:30:00+00:00,NaT,NaT,2
216,SINP-989,2013-10-28 04:40:00+00:00,NaT,NaT,2
217,SINP-990,2013-10-29 07:10:00+00:00,NaT,NaT,2
221,SINP-994,2013-12-08 00:00:00+00:00,NaT,NaT,2
222,SINP-995,2013-12-14 14:00:00+00:00,NaT,NaT,2
226,SINP-999,2014-01-07 14:26:00+00:00,NaT,NaT,2


# Match starts to conjunctions

Use the inclusive rule `abs(event_start − conjunction_Date) <= 24 hours`
for **each** catalog event. Require both MESSENGER and its listed partner to
match the **same daily conjunction row**. This permits at most 48 hours between
the two starts. Each listed partner is evaluated separately, including days
with multiple partners. Repeated daily hits for one event pair are deduplicated
later, while every supporting conjunction row remains in the detailed export.

This is the requested geometric/time screen, not a propagation solution or proof
of a shared solar eruption. Events taking more than 48 hours to travel between
the spacecraft can be missed. No speed gate or nearest-event selection is added.

In [6]:
partner_column = conjunctions.columns[-1]
targets = conjunctions.assign(target=conjunctions[partner_column].str.split(";")).explode("target")
targets["target"] = targets["target"].str.strip()
geometry_columns = {
    "Venus": ("Venus_AU", "Diff_Venus_deg"),
    "Earth": ("Earth_AU", "Diff_Earth_deg"),
    "STEREO_A": ("STEREO_A_AU", "Diff_STEREO_A_deg"),
    "STEREO_B": ("STEREO_B_AU", "Diff_STEREO_B_deg"),
}
assert set(targets["target"]) <= set(geometry_columns)
for target, (radius_column, angle_column) in geometry_columns.items():
    mask = targets["target"].eq(target)
    targets.loc[mask, "partner_radius_au"] = targets.loc[mask, radius_column]
    targets.loc[mask, "angular_separation_deg"] = targets.loc[mask, angle_column]
assert targets["angular_separation_deg"].between(0, 20.01).all()
assert (targets["partner_radius_au"] > targets["Mercury_AU"]).all()
targets = targets[[
    "conjunction_id", "Date", "target", "Mercury_AU", "partner_radius_au", "angular_separation_deg"
]].reset_index(drop=True)

messenger_events = events_in_span[events_in_span["target"].eq("Mercury")].copy()
messenger_matches = conjunctions[["conjunction_id", "Date"]].merge(
    messenger_events[["event_id", "start_time", "end_time"]], how="cross"
).rename(columns={"event_id": "messenger_id", "start_time": "messenger_start", "end_time": "messenger_end"})
messenger_matches["messenger_offset_h"] = (
    messenger_matches["messenger_start"] - messenger_matches["Date"]
).dt.total_seconds() / 3600
messenger_matches = messenger_matches[
    (messenger_matches["messenger_start"] - messenger_matches["Date"]).abs() <= TOLERANCE
].copy()

partner_matches = targets.merge(
    events_in_span[~events_in_span["target"].eq("Mercury")], on="target", how="inner"
).rename(columns={"event_id": "partner_id", "start_time": "partner_start", "end_time": "partner_end"})
partner_matches["partner_offset_h"] = (
    partner_matches["partner_start"] - partner_matches["Date"]
).dt.total_seconds() / 3600
partner_matches = partner_matches[
    (partner_matches["partner_start"] - partner_matches["Date"]).abs() <= TOLERANCE
].copy()

window_pairs = partner_matches.merge(
    messenger_matches.drop(columns="Date"), on="conjunction_id", how="inner", validate="many_to_many"
)
assert (window_pairs["messenger_offset_h"].abs() <= TOLERANCE.total_seconds() / 3600).all()
assert (window_pairs["partner_offset_h"].abs() <= TOLERANCE.total_seconds() / 3600).all()

## Deduplicate event pairs and expose ambiguous associations

Keep the conjunction day minimizing the larger absolute time offset as the
representative geometry; retain the number and range of supporting days.
Do not collapse Wind and SINP rows into a single Earth event. Candidate counts
are within each catalog; reuse of one partner event across MESSENGER events is
also flagged. All alternative candidates remain available.

Chronology is diagnostic: partners are farther from the Sun in the supplied
geometry. A partner MO starting earlier than MESSENGER is flagged for review.
The implied radial speed uses the supplied daily **planet** distances as a
proxy for spacecraft radii and the difference in **MO-start** times; it is not
a measured shock speed. Neither chronology nor speed changes the ±24-hour rule.

In [7]:
pair_key = ["messenger_id", "target", "catalog", "partner_id"]
support = window_pairs.groupby(pair_key, as_index=False).agg(
    supporting_days=("conjunction_id", "size"),
    first_conjunction=("Date", "min"), last_conjunction=("Date", "max"),
)
window_pairs["max_anchor_offset_h"] = window_pairs[["messenger_offset_h", "partner_offset_h"]].abs().max(axis=1)
pairs = window_pairs.sort_values(["max_anchor_offset_h", "Date"]).drop_duplicates(pair_key)
pairs = pairs.merge(support, on=pair_key, validate="one_to_one")
pairs = pairs.sort_values(["messenger_start", "target", "catalog", "partner_start"]).reset_index(drop=True)
pairs.insert(0, "pair_id", [f"PAIR-{i + 1:04d}" for i in range(len(pairs))])
pairs["start_delay_h"] = (pairs["partner_start"] - pairs["messenger_start"]).dt.total_seconds() / 3600
pairs["chronology"] = "partner_after_MESSENGER"
pairs.loc[pairs["start_delay_h"].eq(0), "chronology"] = "simultaneous_starts"
pairs.loc[pairs["start_delay_h"].lt(0), "chronology"] = "partner_before_MESSENGER"
pairs["implied_radial_speed_km_s"] = (
    (pairs["partner_radius_au"] - pairs["Mercury_AU"]) * AU_KM
    / (pairs["start_delay_h"] * 3600).where(pairs["start_delay_h"].gt(0))
)
pairs["candidates_in_same_catalog"] = pairs.groupby(
    ["messenger_id", "target", "catalog"]
)["partner_id"].transform("size")
pairs["messenger_candidates_for_partner"] = pairs.groupby(
    ["target", "catalog", "partner_id"]
)["messenger_id"].transform("nunique")
assert not pairs.duplicated(pair_key).any()
assert (pairs["start_delay_h"].abs() <= 2 * TOLERANCE.total_seconds() / 3600).all()

# Earth: Wind and SINP evidence

Cross-reference Earth candidates only when they have the same MESSENGER event
and their Wind MO start and SINP `T_start` differ by at most 24 hours. This is
a separate, explicit catalog-agreement check. Preserve all alternatives rather
than assuming Wind and SINP are independent observations or forcing a one-to-one
match. The two lists may use the same underlying in-situ evidence.

In [8]:
earth_pairs = pairs[pairs["target"].eq("Earth")]
wind_pairs = earth_pairs[earth_pairs["catalog"].eq("HELIO4CAST")][
    ["messenger_id", "pair_id", "partner_id", "partner_start"]
]
sinp_pairs = earth_pairs[earth_pairs["catalog"].eq("SINP")][
    ["messenger_id", "pair_id", "partner_id", "partner_start"]
]
earth_agreement = wind_pairs.merge(sinp_pairs, on="messenger_id", suffixes=("_wind", "_sinp"))
earth_agreement["sinp_minus_wind_h"] = (
    earth_agreement["partner_start_sinp"] - earth_agreement["partner_start_wind"]
).dt.total_seconds() / 3600
earth_agreement = earth_agreement[
    (earth_agreement["partner_start_sinp"] - earth_agreement["partner_start_wind"]).abs() <= TOLERANCE
].reset_index(drop=True)
supported_pair_ids = set(earth_agreement["pair_id_wind"]) | set(earth_agreement["pair_id_sinp"])
pairs["earth_other_catalog_within_24h"] = pd.Series(pd.NA, index=pairs.index, dtype="boolean")
earth_mask = pairs["target"].eq("Earth")
pairs.loc[earth_mask, "earth_other_catalog_within_24h"] = pairs.loc[earth_mask, "pair_id"].isin(supported_pair_ids)
display(earth_agreement)

,messenger_id,pair_id_wind,partner_id_wind,partner_start_wind,pair_id_sinp,partner_id_sinp,partner_start_sinp,sinp_minus_wind_h
0,ICME_MES_WINSLOW_20140217_01,PAIR-0009,ICME_Wind_NASA_20140215_01,2014-02-16 00:43:00+00:00,PAIR-0011,SINP-1006,2014-02-16 05:00:00+00:00,4.283333
1,ICME_MES_WINSLOW_20140217_01,PAIR-0010,ICME_Wind_NASA_20140218_01,2014-02-18 15:07:00+00:00,PAIR-0013,SINP-1008,2014-02-18 15:00:00+00:00,-0.116667


# Results

Counts below are **candidate event pairs**, not confirmed shared CMEs. Each pair
is counted once regardless of how many daily conjunction rows support it.
Earth catalogs remain separate, so their counts must not be added as distinct
physical CMEs. Review the chronology and ambiguity columns before choosing
events for in-situ inspection.

In [9]:
summary = pairs.groupby(["target", "catalog"], as_index=False).agg(
    candidate_pairs=("pair_id", "size"),
    messenger_events=("messenger_id", "nunique"),
    partner_events=("partner_id", "nunique"),
    outward_order=("start_delay_h", lambda values: values.gt(0).sum()),
    reversed_order=("start_delay_h", lambda values: values.lt(0).sum()),
    ambiguous_pairs=("candidates_in_same_catalog", lambda values: values.gt(1).sum()),
)
display(summary)

,target,catalog,candidate_pairs,messenger_events,partner_events,outward_order,reversed_order,ambiguous_pairs
0,Earth,HELIO4CAST,4,3,4,3,1,2
1,Earth,SINP,4,2,4,1,3,3
2,STEREO_A,HELIO4CAST,1,1,1,0,1,0
3,STEREO_B,HELIO4CAST,1,1,1,1,0,0
4,Venus,HELIO4CAST,4,4,3,4,0,0


In [10]:
result_columns = [
    "pair_id", "messenger_id", "messenger_start", "target", "catalog", "partner_id", "partner_start",
    "Date", "messenger_offset_h", "partner_offset_h", "start_delay_h", "chronology",
    "angular_separation_deg", "implied_radial_speed_km_s", "supporting_days",
    "candidates_in_same_catalog", "messenger_candidates_for_partner", "earth_other_catalog_within_24h",
]
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(pairs[result_columns])

,pair_id,messenger_id,messenger_start,target,catalog,partner_id,partner_start,Date,messenger_offset_h,partner_offset_h,start_delay_h,chronology,angular_separation_deg,implied_radial_speed_km_s,supporting_days,candidates_in_same_catalog,messenger_candidates_for_partner,earth_other_catalog_within_24h
0,PAIR-0001,ICME_MES_WINSLOW_20110604_01,2011-06-04 16:15:00+00:00,Venus,HELIO4CAST,ICME_VEX_SGOOD_20110605_01,2011-06-05 15:12:00+00:00,2011-06-05 00:00:00+00:00,-7.750000,15.200000,22.950000,partner_after_MESSENGER,6.57,726.080200,1,1,2,<NA>
1,PAIR-0002,ICME_MES_WINSLOW_20110605_01,2011-06-05 04:29:00+00:00,Venus,HELIO4CAST,ICME_VEX_SGOOD_20110605_01,2011-06-05 15:12:00+00:00,2011-06-05 00:00:00+00:00,4.483333,15.200000,10.716667,partner_after_MESSENGER,6.57,1554.918252,2,1,2,<NA>
2,PAIR-0003,ICME_MES_WINSLOW_20111015_01,2011-10-15 11:13:00+00:00,Venus,HELIO4CAST,ICME_VEX_SGOOD_20111016_01,2011-10-16 06:05:00+00:00,2011-10-16 00:00:00+00:00,-12.783333,6.083333,18.866667,partner_after_MESSENGER,0.73,581.475823,1,1,1,<NA>
3,PAIR-0004,ICME_MES_WINSLOW_20120102_01,2012-01-02 19:54:00+00:00,STEREO_A,HELIO4CAST,ICME_STEREO_A_JIAN_20120101_01,2012-01-01 17:00:00+00:00,2012-01-02 00:00:00+00:00,19.900000,-7.000000,-26.900000,partner_before_MESSENGER,2.43,NaN,1,1,1,<NA>
4,PAIR-0005,ICME_MES_WINSLOW_20120307_01,2012-03-07 06:10:00+00:00,Venus,HELIO4CAST,ICME_VEX_SGOOD_20120307_01,2012-03-07 20:13:00+00:00,2012-03-08 00:00:00+00:00,-17.833333,-3.783333,14.050000,partner_after_MESSENGER,2.82,1186.017124,2,1,1,<NA>
5,PAIR-0006,ICME_MES_WINSLOW_20130711_01,2013-07-11 01:57:00+00:00,Earth,HELIO4CAST,ICME_Wind_NASA_20130712_01,2013-07-12 20:24:00+00:00,2013-07-12 00:00:00+00:00,-22.050000,20.400000,42.450000,partner_after_MESSENGER,4.58,560.918596,1,1,1,False
6,PAIR-0007,ICME_MES_WINSLOW_20130711_01,2013-07-11 01:57:00+00:00,Earth,SINP,SINP-977,2013-07-10 20:00:00+00:00,2013-07-11 00:00:00+00:00,1.950000,-4.000000,-5.950000,partner_before_MESSENGER,2.49,NaN,1,1,1,False
7,PAIR-0008,ICME_MES_WINSLOW_20131029_01,2013-10-29 11:14:00+00:00,Earth,HELIO4CAST,ICME_Wind_NASA_20131029_01,2013-10-30 18:14:00+00:00,2013-10-30 00:00:00+00:00,-12.766667,18.233333,31.000000,partner_after_MESSENGER,13.23,887.399556,1,1,1,False
8,PAIR-0009,ICME_MES_WINSLOW_20140217_01,2014-02-17 07:02:00+00:00,Earth,HELIO4CAST,ICME_Wind_NASA_20140215_01,2014-02-16 00:43:00+00:00,2014-02-17 00:00:00+00:00,7.033333,-23.283333,-30.316667,partner_before_MESSENGER,4.57,NaN,1,2,1,True
9,PAIR-0010,ICME_MES_WINSLOW_20140217_01,2014-02-17 07:02:00+00:00,Earth,HELIO4CAST,ICME_Wind_NASA_20140218_01,2014-02-18 15:07:00+00:00,2014-02-18 00:00:00+00:00,-16.966667,15.116667,32.083333,partner_after_MESSENGER,8.36,815.988386,1,2,1,True


## Opportunities with no partner match

Keep every MESSENGER/conjunction/target opportunity, even when the partner has
no matched catalog entry. A missing counterpart is a catalog non-match, not an
observational nondetection. Dates without a MESSENGER start match are not
opportunities in this table.

In [11]:
opportunities = targets.merge(messenger_matches.drop(columns="Date"), on="conjunction_id")
counts = window_pairs.groupby(["conjunction_id", "messenger_id", "target"]).size().rename("partner_catalog_entries")
opportunities = opportunities.merge(counts, on=["conjunction_id", "messenger_id", "target"], how="left")
opportunities["partner_catalog_entries"] = opportunities["partner_catalog_entries"].fillna(0).astype(int)
messenger_status = messenger_events[["event_id", "start_time"]].rename(columns={"event_id": "messenger_id"})
messenger_status["within_24h_of_any_conjunction"] = messenger_status["messenger_id"].isin(messenger_matches["messenger_id"])
messenger_status["has_partner_candidate"] = messenger_status["messenger_id"].isin(pairs["messenger_id"])
unmatched = opportunities.groupby(["messenger_id", "target"], as_index=False).agg(
    first_conjunction=("Date", "min"), last_conjunction=("Date", "max"),
    partner_catalog_entries=("partner_catalog_entries", "sum"),
)
display(unmatched[unmatched["partner_catalog_entries"].eq(0)])

,messenger_id,target,first_conjunction,last_conjunction,partner_catalog_entries
0,ICME_MES_MOESTL_20141214_01,STEREO_A,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
1,ICME_MES_MOESTL_20141214_01,STEREO_B,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
2,ICME_MES_MOESTL_20141214_01,Venus,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
3,ICME_MES_MOESTL_20141214_02,STEREO_A,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
4,ICME_MES_MOESTL_20141214_02,STEREO_B,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
5,ICME_MES_MOESTL_20141214_02,Venus,2014-12-14 00:00:00+00:00,2014-12-15 00:00:00+00:00,0
6,ICME_MES_MOESTL_20150407_01,STEREO_A,2015-04-08 00:00:00+00:00,2015-04-09 00:00:00+00:00,0
7,ICME_MES_MOESTL_20150407_01,STEREO_B,2015-04-08 00:00:00+00:00,2015-04-09 00:00:00+00:00,0
8,ICME_MES_WINSLOW_20110519_01,STEREO_A,2011-05-19 00:00:00+00:00,2011-05-20 00:00:00+00:00,0
11,ICME_MES_WINSLOW_20110608_01,Venus,2011-06-08 00:00:00+00:00,2011-06-08 00:00:00+00:00,0


## Compare with the earlier Earth workbook selection

Use the workbook's stable SINP `index` to compare against the local SINP snapshot.
The older selection can contain shock-only rows and can cover an Earth ICME
without a matching MESSENGER ICME in HELIO4CAST; do not interpret membership in
that sheet as a confirmed two-spacecraft event.

In [12]:
prior_comparison = prior_earth[["index", "T_start", "T_shock", "T_gen"]].copy()
prior_comparison["event_id"] = "SINP-" + prior_comparison["index"].astype(str)
prior_comparison = prior_comparison.merge(
    sinp[["event_id", "T_start", "T_shock"]], on="event_id", how="left",
    suffixes=("_workbook", "_snapshot"), validate="one_to_one", indicator=True,
)
prior_comparison["start_time_agrees"] = (
    prior_comparison["T_start_workbook"].eq(prior_comparison["T_start_snapshot"])
    | (prior_comparison["T_start_workbook"].isna() & prior_comparison["T_start_snapshot"].isna())
)
prior_comparison["matched_to_MESSENGER_now"] = prior_comparison["event_id"].isin(
    pairs.loc[pairs["catalog"].eq("SINP"), "partner_id"]
)
display(prior_comparison)

,index,T_start_workbook,T_shock_workbook,T_gen,event_id,T_start_snapshot,T_shock_snapshot,_merge,start_time_agrees,matched_to_MESSENGER_now
0,838,2011-04-09 18:00:00+00:00,NaT,2011-04-09 18:00:00+00:00,SINP-838,2011-04-09 18:00:00+00:00,NaT,both,True,False
1,882,2011-12-02 18:00:00+00:00,2011-12-02 17:17:00+00:00,2011-12-02 17:17:00+00:00,SINP-882,2011-12-02 18:00:00+00:00,2011-12-02 17:17:00+00:00,both,True,False
2,883,2011-12-04 08:00:00+00:00,NaT,2011-12-04 08:00:00+00:00,SINP-883,2011-12-04 08:00:00+00:00,NaT,both,True,False
3,901,2012-03-12 22:00:00+00:00,2012-03-12 06:00:00+00:00,2012-03-12 06:00:00+00:00,SINP-901,2012-03-12 22:00:00+00:00,2012-03-12 06:00:00+00:00,both,True,False
4,902,2012-03-15 17:00:00+00:00,2012-03-15 13:06:00+00:00,2012-03-15 13:06:00+00:00,SINP-902,2012-03-15 17:00:00+00:00,2012-03-15 13:06:00+00:00,both,True,False
5,903,2012-03-22 08:00:00+00:00,NaT,2012-03-22 08:00:00+00:00,SINP-903,2012-03-22 08:00:00+00:00,NaT,both,True,False
6,922,2012-07-22 11:00:00+00:00,2012-07-21 16:12:00+00:00,2012-07-21 16:12:00+00:00,SINP-922,2012-07-22 11:00:00+00:00,2012-07-21 16:12:00+00:00,both,True,False
7,923,2012-07-26 12:00:00+00:00,NaT,2012-07-26 12:00:00+00:00,SINP-923,2012-07-26 12:00:00+00:00,NaT,both,True,False
8,938,2012-11-13 06:00:00+00:00,2012-11-12 15:00:00+00:00,2012-11-12 15:00:00+00:00,SINP-938,2012-11-13 06:00:00+00:00,2012-11-12 15:00:00+00:00,both,True,False
9,939,2012-11-13 08:00:00+00:00,2012-11-12 23:11:00+00:00,2012-11-12 23:11:00+00:00,SINP-939,2012-11-13 08:00:00+00:00,2012-11-12 23:11:00+00:00,both,True,False


# Validation and exports

Independently enumerate every permitted daily MESSENGER–partner pairing using
the scalar inclusive ±24-hour rule and compare with the vectorized join.
Check the exact tolerance endpoints and just-outside values, the date-coverage
boundary, and all saved input hashes. Export every candidate and non-match,
including the detailed daily associations, so the selection is auditable.

In [13]:
boundary_offsets = pd.to_timedelta([-24.0001, -24, 0, 24, 24.0001], unit="h")
assert list(abs(boundary_offsets) <= TOLERANCE) == [False, True, True, True, False]
reference_pairs = set()
for row in targets.itertuples(index=False):
    matching_messenger = messenger_events[
        (messenger_events["start_time"] - row.Date).abs() <= TOLERANCE
    ]
    matching_partner = events_in_span[
        events_in_span["target"].eq(row.target)
        & ((events_in_span["start_time"] - row.Date).abs() <= TOLERANCE)
    ]
    for messenger in matching_messenger.itertuples(index=False):
        for partner in matching_partner.itertuples(index=False):
            reference_pairs.add((row.conjunction_id, messenger.event_id, row.target, partner.catalog, partner.event_id))
actual_pairs = set(window_pairs[["conjunction_id"] + pair_key].itertuples(index=False, name=None))
assert reference_pairs == actual_pairs
assert len(actual_pairs) == len(window_pairs)
assert len(pairs) == len({key[1:] for key in reference_pairs})
assert sinp_events.loc[sinp["T_start"].isna(), "start_time"].isna().all()
assert not set(sinp_missing_start["event_id"]) & set(pairs.loc[pairs["catalog"].eq("SINP"), "partner_id"])
display(Markdown(f"**Validated:** {len(reference_pairs)} daily pairings, {len(pairs)} unique candidate pairs; "
                 "all start-time, missingness, source-hash, and boundary checks passed."))

**Validated:** 17 daily pairings, 14 unique candidate pairs; all start-time, missingness, source-hash, and boundary checks passed.

In [14]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    "candidate_pairs.csv": pairs,
    "daily_pair_matches.csv": window_pairs,
    "catalog_summary.csv": summary,
    "earth_catalog_agreement.csv": earth_agreement,
    "messenger_conjunction_opportunities.csv": opportunities,
    "messenger_event_status.csv": messenger_status,
    "sinp_missing_start.csv": sinp_missing_start,
    "earlier_earth_selection_comparison.csv": prior_comparison,
}
for filename, table in exports.items():
    table.to_csv(RESULTS_DIR / filename, index=False)
analysis_manifest = {
    "tolerance_hours": TOLERANCE.total_seconds() / 3600,
    "matching_rule": "Both catalog starts within inclusive tolerance of the same supplied daily conjunction timestamp",
    "helio4cast_start_field": "mo_start_time",
    "sinp_start_field": "T_start",
    "naive_input_timezone_assumption": "UTC",
    "source_catalog_url": catalog_manifest["catalog_url"],
    "input_sha256": {name: metadata["sha256"] for name, metadata in input_manifest.items()},
    "helio4cast_sha256": catalog_manifest["sha256"],
    "candidate_pairs": len(pairs),
    "daily_pairings": len(window_pairs),
    "output_sha256": {name: hashlib.sha256((RESULTS_DIR / name).read_bytes()).hexdigest() for name in exports},
}
(RESULTS_DIR / "analysis_metadata.json").write_text(json.dumps(analysis_manifest, indent=2) + "\n")
display(pd.DataFrame({"export": list(exports), "rows": [len(table) for table in exports.values()]}))

,export,rows
0,candidate_pairs.csv,14
1,daily_pair_matches.csv,17
2,catalog_summary.csv,5
3,earth_catalog_agreement.csv,2
4,messenger_conjunction_opportunities.csv,80
5,messenger_event_status.csv,64
6,sinp_missing_start.csv,22
7,earlier_earth_selection_comparison.csv,33


## Interpretation

The next cell reports observed counts from this run. Candidate associations need
in-situ/profile and solar-source review before being called the same CME.
The daily geometry and ±24-hour window are the only matching criteria. The cone
is evaluated at the supplied midnight geometry, which may differ from event-time
geometry. Missing coverage, catalog selection differences, delayed propagation,
and MO boundary choices can all produce non-matches.

In [15]:
display(Markdown(
    f"The {len(conjunctions)} daily conjunction rows yield **{len(pairs)} distinct catalog candidate pairs**, "
    f"involving **{pairs['messenger_id'].nunique()} MESSENGER events**. "
    f"Of the pairs, **{pairs['start_delay_h'].gt(0).sum()}** have the outer partner starting later, "
    f"**{pairs['start_delay_h'].lt(0).sum()}** have reversed start order, and "
    f"**{pairs['start_delay_h'].eq(0).sum()}** have simultaneous starts. "
    f"Earth has **{earth_agreement['messenger_id'].nunique()} MESSENGER events** with compatible starts in both Wind and SINP. "
    f"**{len(sinp_missing_start)} SINP shock entries** in the overall table span lack an ICME start and were not matched. "
    "These counts describe catalog candidates under the specified tolerance, not confirmed common eruptions."
))

The 605 daily conjunction rows yield **14 distinct catalog candidate pairs**, involving **9 MESSENGER events**. Of the pairs, **9** have the outer partner starting later, **5** have reversed start order, and **0** have simultaneous starts. Earth has **1 MESSENGER events** with compatible starts in both Wind and SINP. **22 SINP shock entries** in the overall table span lack an ICME start and were not matched. These counts describe catalog candidates under the specified tolerance, not confirmed common eruptions.